# 14장. sklearn으로 검증하기

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 13장 |
| 이 장의 결과물 | 검증 함수 (`src/verify.py`), 검증 로그, README에 쓸 sklearn 설정 문장 |
| 과제 연결 | "Scikit-learn TfidfVectorizer와 비교하여 오차가 1e-6 이내인지 검증한다. 어떤 설정으로 검증했는지 README에 명시한다" |

**학습 목표**
- `TfidfVectorizer`의 주요 파라미터를 직접 구현의 설정과 짝지을 수 있다.
- 두 구현이 **같은 토큰**을 보도록 sklearn을 설정할 수 있다.
- 어휘 → DF/IDF → 행렬 순서로 비교하고, 최대·평균 오차로 판정할 수 있다.
- 여러 설정 조합과 데이터로 검증 로그를 만들 수 있다.
- 검증이 실패했을 때 어긋난 위치를 찾아 원인을 좁힐 수 있다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

# ── 8~10장에서 만든 전처리 파이프라인 (src/preprocess.py로 옮길 내용) ──
from konlpy.tag import Okt
from nltk.corpus import stopwords

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}
EN_STOP = frozenset(stopwords.words("english"))
DOMAIN_STOPWORDS_EN = frozenset({
    "would", "could", "might", "may", "also", "like", "get", "got", "go", "going", "know", "think",
    "see", "say", "make", "want", "look", "one", "well", "much", "many", "even", "way", "something",
    "anyone", "since", "still", "really", "right", "another", "cant",
})
STOPWORDS_KO = frozenset({
    "하다", "되다", "있다", "없다", "이다", "않다",
    "등", "것", "수", "중", "및", "때", "후", "내", "듯", "제", "개", "의", "로", "종합",
})
_okt = Okt()


def clean_en(text: str) -> str:
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


class EnglishPreprocessor:
    def __init__(self, use_domain_stopwords: bool = True, min_len: int = 2, max_len: int = 20):
        self.stopwords = EN_STOP | (DOMAIN_STOPWORDS_EN if use_domain_stopwords else frozenset())
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for tok in clean_en(text).split():
            if tok in self.stopwords:
                continue
            tok = tok.removesuffix("'s").replace("'", "")
            if self.min_len <= len(tok) <= self.max_len and tok not in self.stopwords:
                tokens.append(tok)
        return tokens


class KoreanPreprocessor:
    def __init__(self, tokenizer: str = "okt", use_stopwords: bool = True):
        if tokenizer not in ("okt", "whitespace"):
            raise ValueError(f"tokenizer는 'okt' 또는 'whitespace': {tokenizer}")
        self.tokenizer = tokenizer
        self.stopwords = STOPWORDS_KO if use_stopwords else frozenset()

    def tokenize(self, text: str) -> list[str]:
        if self.tokenizer == "whitespace":
            return clean_ko(text).lower().split()
        return [w.lower() for w, p in _okt.pos(clean_ko(text), norm=True, stem=True)
                if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]

    def __call__(self, text: str) -> list[str]:
        return [t for t in self.tokenize(text) if t not in self.stopwords and not t.isdigit()]

# ── 로드 + 전처리를 한 번에 (src/corpus.py로 옮길 내용) ──
MIN_TOKENS = {"en": 5, "ko": 2}   # 전처리 후 토큰이 이보다 적은 문서는 뺀다 (짧은 답글, 주제어 없는 제목)
MIN_DF = {"en": 3, "ko": 2}       # 어휘에 넣을 최소 문서 빈도 (11장)


def load_corpus(lang: str, tokenizer: str = "okt") -> pd.DataFrame:
    """열: text, label, tokens. lang은 "en" 또는 "ko"."""
    df = load_english() if lang == "en" else load_korean()
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor(tokenizer)
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df

# ── 13장에서 만든 NumPy TF-IDF (src/tfidf.py로 옮길 내용) ──
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)

# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

import numpy as np
import pandas as pd

STOP = {"the", "a", "into", "our", "in", "new"}
FILTERED = [[t for t in doc.lower().strip(".!").split() if t not in STOP] for doc in MINI_DOCS]
np.set_printoptions(precision=4, suppress=True, linewidth=120)

import platform
import tempfile
import time

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer

## 14.1 TfidfVectorizer 개요

`TfidfVectorizer`는 **원문 문자열**을 받아 내부에서 전처리·토큰화까지 한 뒤 TF-IDF 희소 행렬을 돌려줍니다. 기본 설정 그대로 쓰면 **우리 전처리와 다른 토큰**을 만듭니다.

In [2]:
default = TfidfVectorizer().fit(MINI_DOCS)
print("sklearn 기본 토큰화 어휘:", list(default.get_feature_names_out()))
print("우리 전처리 어휘        :", sorted({t for d in FILTERED for t in d}))

sklearn 기본 토큰화 어휘: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'in', 'into', 'launched', 'nasa', 'new', 'our', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'the', 'threw', 'watched', 'won']
우리 전처리 어휘        : ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']


기본값은 소문자 변환 후 정규식 `(?u)\b\w\w+\b`로 2글자 이상 단어를 뽑고, 불용어는 지우지 않습니다. 그래서 `the`, `into`, `new` 같은 단어가 남고 한 글자 `a`는 빠집니다. **토큰이 다르면 어휘가 다르고, 어휘가 다르면 비교 자체가 의미가 없습니다.**

## 14.2 파라미터 대응표

| 직접 구현 `NumpyTfidf` | sklearn `TfidfVectorizer` | 비고 |
|---|---|---|
| (입력이 이미 토큰 리스트) | `analyzer=identity` | 토큰화를 건너뛰고 우리 토큰을 그대로 쓴다 |
| | `lowercase=False` | 소문자 변환은 이미 전처리에서 했다 |
| `tf="raw"` / `tf="log"` | `sublinear_tf=False` / `True` | |
| `smooth_idf=True/False` | `smooth_idf=True/False` | 같은 식 |
| `norm="l2"` / `None` | `norm="l2"` / `None` | |
| `min_df=3` | `min_df=3` | 정수 = 문서 수 |
| `max_df=1.0` | `max_df=1.0` | 실수 = 비율 |
| `np.float64` | `dtype=np.float64` | sklearn 기본도 float64 |
| `vocabulary_`, `idf_` | `vocabulary_`, `idf_`, `get_feature_names_out()` | 이름까지 같게 지었다 |

## 14.3 같은 토큰을 보게 하기: analyzer

`analyzer`에 **함수**를 주면 sklearn은 문서마다 그 함수를 호출해 토큰 리스트를 얻고, 자체 소문자 변환·정규식·불용어 처리를 **전혀 하지 않습니다.** 입력이 이미 토큰 리스트이므로 그대로 돌려주는 함수를 줍니다.

```python
def identity(doc):
    return doc
```

> `lambda doc: doc`도 동작하지만, 이름 있는 함수를 쓰면 객체를 저장(pickle)할 때 문제가 없고 로그에 설정을 적기도 쉽습니다.
>
> 다른 방법으로 `tokenizer=identity, preprocessor=identity, token_pattern=None, lowercase=False`를 함께 줄 수도 있습니다. 하나라도 빠뜨리면 sklearn이 다시 손을 대므로 `analyzer` 하나로 끝내는 쪽이 안전합니다.

In [3]:
def identity(doc):
    return doc


sk = TfidfVectorizer(analyzer=identity, lowercase=False).fit(FILTERED)
ours = NumpyTfidf().fit(FILTERED)
print("sklearn 어휘:", list(sk.get_feature_names_out()))
print("어휘 순서까지 같은가:", list(sk.get_feature_names_out()) == list(ours.vocabulary_))

sklearn 어휘: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']
어휘 순서까지 같은가: True


## 14.4 비교 순서: 어휘 → IDF → 행렬

행렬부터 비교하면, 틀렸을 때 **왜** 틀렸는지 알 수 없습니다. 앞 단계부터 하나씩 확인합니다.

1. **어휘**: 두 어휘 집합이 같은가? 다르면 행렬 비교는 의미가 없으므로 바로 FAIL.
2. **열 맞추기**: 두 구현 모두 정렬하므로 순서가 같아야 하지만, 안전하게 **단어 → 열 번호 사전으로 sklearn 열을 다시 배치**한 뒤 비교합니다.
3. **IDF**: `idf_` 벡터 비교. DF가 다르면 여기서 드러납니다.
4. **0이 아닌 위치**: 같은 칸에 값이 있는가?
5. **값**: 최대·평균 오차.

## 14.5 toarray(): 희소 행렬을 밀집 행렬로

sklearn은 0이 아닌 칸만 저장하는 **희소 행렬**(`scipy.sparse`)을 돌려줍니다. NumPy 배열과 빼기를 하려면 `.toarray()`로 바꿉니다.

In [4]:
X_sparse = sk.transform(FILTERED)
print(type(X_sparse).__name__, X_sparse.shape, "| 0이 아닌 칸:", X_sparse.nnz)
print(type(X_sparse.toarray()).__name__, X_sparse.toarray().shape)

csr_matrix (5, 15) | 0이 아닌 칸: 20
ndarray (5, 15)


## 14.6 검증 함수

위 다섯 단계를 함수 하나로 만듭니다. **이 셀이 `src/verify.py`의 핵심**입니다. 영어 행렬은 약 500MB라서 두 행렬과 오차 행렬이 동시에 메모리에 올라가지 않도록 sklearn 결과에서 바로 빼서 오차를 구합니다.

In [5]:
TOLERANCE = 1e-6


def compare(docs, ours_cfg: dict, sk_cfg: dict, min_df: int, vectorizer_cls=NumpyTfidf) -> dict:
    ours = vectorizer_cls(min_df=min_df, **ours_cfg)
    x_ours = ours.fit_transform(docs)

    sk = TfidfVectorizer(analyzer=identity, lowercase=False, min_df=min_df, dtype=np.float64, **sk_cfg)
    x_sk_sparse = sk.fit_transform(docs)
    sk_terms = sk.get_feature_names_out()

    # ① 어휘
    only_sk = set(sk_terms) - set(ours.vocabulary_)
    only_ours = set(ours.vocabulary_) - set(sk_terms)
    if only_sk or only_ours:
        return {"shape": x_ours.shape, "same_vocab": False,
                "detail": f"sklearn에만 {len(only_sk)}개, 직접 구현에만 {len(only_ours)}개",
                "idf_max_err": np.nan, "max_err": np.nan, "mean_err": np.nan, "nnz_equal": False, "pass": False}

    # ② 열 맞추기: sklearn 열 j → 우리 열 order[j]
    order = np.array([ours.vocabulary_[t] for t in sk_terms])
    x_sk = np.zeros_like(x_ours)
    x_sk[:, order] = x_sk_sparse.toarray()
    idf_sk = np.empty_like(ours.idf_)
    idf_sk[order] = sk.idf_

    # ③ IDF, ④ 0이 아닌 위치, ⑤ 값
    nnz_equal = bool(((x_ours != 0) == (x_sk != 0)).all())
    np.subtract(x_sk, x_ours, out=x_sk)          # 제자리 빼기: 메모리를 아낀다
    np.abs(x_sk, out=x_sk)
    max_err, mean_err = float(x_sk.max()), float(x_sk.mean())
    return {"shape": x_ours.shape, "same_vocab": True, "detail": "",
            "idf_max_err": float(np.abs(ours.idf_ - idf_sk).max()),
            "max_err": max_err, "mean_err": mean_err, "nnz_equal": nnz_equal,
            "pass": max_err <= TOLERANCE}


r = compare(FILTERED, {"tf": "raw", "smooth_idf": True, "norm": "l2"},
            {"sublinear_tf": False, "smooth_idf": True, "norm": "l2"}, min_df=1)
r

{'shape': (5, 15),
 'same_vocab': True,
 'detail': '',
 'idf_max_err': 0.0,
 'max_err': 0.0,
 'mean_err': 0.0,
 'nnz_equal': True,
 'pass': True}

### 4가지 설정 × 3가지 데이터

기본 설정 하나만 맞으면 우연일 수 있으므로, 수식의 각 부분을 바꿔 가며 확인합니다.

| 검증 | 직접 구현 | sklearn | 무엇을 확인하나 |
|---|---|---|---|
| 1 (기본) | `tf="raw", smooth_idf=True, norm="l2"` | `sublinear_tf=False, smooth_idf=True, norm="l2"` | 과제 기본 정의 |
| 2 | `tf="log"` | `sublinear_tf=True` | sublinear TF 식 |
| 3 | `smooth_idf=False` | `smooth_idf=False` | no-smooth IDF 식 |
| 4 | `norm=None` | `norm=None` | 정규화 전 값 (IDF와 TF 곱이 정확한가) |

데이터는 영어, 한국어 Okt, 한국어 공백 토큰화 세 가지입니다. 전체 실행에 1~2분 걸립니다.

In [6]:
CONFIGS = [
    ({"tf": "raw", "smooth_idf": True, "norm": "l2"}, {"sublinear_tf": False, "smooth_idf": True, "norm": "l2"}),
    ({"tf": "log", "smooth_idf": True, "norm": "l2"}, {"sublinear_tf": True, "smooth_idf": True, "norm": "l2"}),
    ({"tf": "raw", "smooth_idf": False, "norm": "l2"}, {"sublinear_tf": False, "smooth_idf": False, "norm": "l2"}),
    ({"tf": "raw", "smooth_idf": True, "norm": None}, {"sublinear_tf": False, "smooth_idf": True, "norm": None}),
]

datasets = {
    "en": (load_corpus("en"), MIN_DF["en"]),
    "ko-okt": (load_corpus("ko", "okt"), MIN_DF["ko"]),
    "ko-whitespace": (load_corpus("ko", "whitespace"), MIN_DF["ko"]),
}

results = {}
t0 = time.perf_counter()
for name, (corpus, min_df) in datasets.items():
    docs = corpus["tokens"].tolist()
    for k, (ours_cfg, sk_cfg) in enumerate(CONFIGS, start=1):
        results[(name, k)] = compare(docs, ours_cfg, sk_cfg, min_df)
print(f"검증 {len(results)}개 완료 ({time.perf_counter() - t0:.0f}초)")

summary = pd.DataFrame([
    {"데이터": name, "검증": k, "shape": r["shape"], "어휘 일치": r["same_vocab"], "0 위치 일치": r["nnz_equal"],
     "IDF 최대 오차": f"{r['idf_max_err']:.2e}", "최대 오차": f"{r['max_err']:.2e}",
     "평균 오차": f"{r['mean_err']:.2e}", "결과": "PASS" if r["pass"] else "FAIL"}
    for (name, k), r in results.items()
])
summary

검증 12개 완료 (12초)


,데이터,검증,shape,어휘 일치,0 위치 일치,IDF 최대 오차,최대 오차,평균 오차,결과
0,en,1,"(4646, 13387)",True,True,0.00e+00,1.55e-15,4.80e-20,PASS
1,en,2,"(4646, 13387)",True,True,0.00e+00,3.05e-16,5.63e-20,PASS
2,en,3,"(4646, 13387)",True,True,0.00e+00,1.11e-15,5.23e-20,PASS
3,en,4,"(4646, 13387)",True,True,0.00e+00,0.00e+00,0.00e+00,PASS
4,ko-okt,1,"(7000, 6332)",True,True,0.00e+00,2.22e-16,1.56e-20,PASS
5,ko-okt,2,"(7000, 6332)",True,True,0.00e+00,2.22e-16,1.54e-20,PASS
6,ko-okt,3,"(7000, 6332)",True,True,0.00e+00,2.22e-16,1.33e-20,PASS
7,ko-okt,4,"(7000, 6332)",True,True,0.00e+00,0.00e+00,0.00e+00,PASS
8,ko-whitespace,1,"(6998, 6344)",True,True,0.00e+00,2.22e-16,9.88e-21,PASS
9,ko-whitespace,2,"(6998, 6344)",True,True,0.00e+00,2.22e-16,9.94e-21,PASS


12개 검증이 모두 PASS이고, 오차는 1e-15 이하입니다. 이것은 **부동소수점 반올림 오차**입니다. 같은 수식이라도 덧셈 순서가 다르면 마지막 자리가 조금 달라질 수 있기 때문에 0이 아닌 값이 나옵니다. 검증 4(`norm=None`)는 나눗셈이 없어 오차가 0입니다.

### 검증 로그 저장
과제 결과 예시와 같은 형식으로 로그를 만들어 파일로 저장합니다. 실제 프로젝트에서는 `outputs/<데이터>/verify_log.txt`에 저장하고, 여기서는 임시 폴더에 저장합니다.

In [7]:
def verify_log(name: str, n_docs: int, min_df: int) -> str:
    lines = [
        "=== TF-IDF 구현 검증 결과 ===",
        f"데이터: {name}, 문서 {n_docs:,}건",
        f"환경: Python {platform.python_version()}, NumPy {np.__version__}, scikit-learn {sklearn.__version__}",
        f"공통 sklearn 설정: TfidfVectorizer(analyzer=<전처리된 토큰 그대로>, lowercase=False, min_df={min_df}, dtype=float64)",
        "",
    ]
    for k, (ours_cfg, sk_cfg) in enumerate(CONFIGS, start=1):
        r = results[(name, k)]
        lines += [
            f"[검증 {k}: {'기본 설정' if k == 1 else '추가 설정'}] 직접 구현 vs Scikit-learn",
            f"  - 직접 구현 설정: NumpyTfidf({', '.join(f'{a}={b!r}' for a, b in ours_cfg.items())}, min_df={min_df})",
            f"  - sklearn 설정 : {', '.join(f'{a}={b!r}' for a, b in sk_cfg.items())}",
            f"  - TF-IDF 행렬 shape: {r['shape']}, 어휘 일치: {r['same_vocab']}, 0이 아닌 위치 일치: {r['nnz_equal']}",
            f"  - IDF 최대 오차: {r['idf_max_err']:.2e}",
            f"  - 최대 오차: {r['max_err']:.2e}",
            f"  - 평균 오차: {r['mean_err']:.2e}",
            f"  - 결과: {'PASS' if r['pass'] else 'FAIL'} (허용 오차 {TOLERANCE:g} 이내)",
            "",
        ]
    return "\n".join(lines)


log_dir = Path(tempfile.mkdtemp())
for name, (corpus, min_df) in datasets.items():
    path = log_dir / name / "verify_log.txt"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(verify_log(name, len(corpus), min_df), encoding="utf-8")
print("저장:", sorted(str(p.relative_to(log_dir)) for p in log_dir.rglob("*.txt")))
print()
print(verify_log("en", len(datasets["en"][0]), MIN_DF["en"]))

저장: ['en/verify_log.txt', 'ko-okt/verify_log.txt', 'ko-whitespace/verify_log.txt']

=== TF-IDF 구현 검증 결과 ===
데이터: en, 문서 4,646건
환경: Python 3.13.5, NumPy 2.5.3, scikit-learn 1.9.1
공통 sklearn 설정: TfidfVectorizer(analyzer=<전처리된 토큰 그대로>, lowercase=False, min_df=3, dtype=float64)

[검증 1: 기본 설정] 직접 구현 vs Scikit-learn
  - 직접 구현 설정: NumpyTfidf(tf='raw', smooth_idf=True, norm='l2', min_df=3)
  - sklearn 설정 : sublinear_tf=False, smooth_idf=True, norm='l2'
  - TF-IDF 행렬 shape: (4646, 13387), 어휘 일치: True, 0이 아닌 위치 일치: True
  - IDF 최대 오차: 0.00e+00
  - 최대 오차: 1.55e-15
  - 평균 오차: 4.80e-20
  - 결과: PASS (허용 오차 1e-06 이내)

[검증 2: 추가 설정] 직접 구현 vs Scikit-learn
  - 직접 구현 설정: NumpyTfidf(tf='log', smooth_idf=True, norm='l2', min_df=3)
  - sklearn 설정 : sublinear_tf=True, smooth_idf=True, norm='l2'
  - TF-IDF 행렬 shape: (4646, 13387), 어휘 일치: True, 0이 아닌 위치 일치: True
  - IDF 최대 오차: 0.00e+00
  - 최대 오차: 3.05e-16
  - 평균 오차: 5.63e-20
  - 결과: PASS (허용 오차 1e-06 이내)

[검증 3: 추가 설정] 직접 구현 vs Scikit-learn
  - 직접 구현 설정: Numpy

## 14.7 검증이 실패할 때: 디버깅 실습

일부러 틀린 구현 두 개를 만들고, 검증이 어떻게 실패하는지, 어디서 원인을 찾는지 봅니다. 빠르게 돌리려고 영어 앞 500건을 씁니다.

### 버그 ① IDF 끝의 +1을 빠뜨림

In [8]:
class NoTailOneTfidf(NumpyTfidf):
    def fit(self, docs):
        super().fit(docs)
        self.idf_ = self.idf_ - 1.0          # 버그: ln((1+n)/(1+df)) 만 쓴 셈
        return self


small = datasets["en"][0]["tokens"].tolist()[:500]
base_cfg = ({"tf": "raw", "smooth_idf": True, "norm": "l2"}, {"sublinear_tf": False, "smooth_idf": True, "norm": "l2"})
r = compare(small, *base_cfg, min_df=3, vectorizer_cls=NoTailOneTfidf)
print({k: r[k] for k in ["same_vocab", "idf_max_err", "max_err", "pass"]})

{'same_vocab': True, 'idf_max_err': 1.0, 'max_err': 0.08928557862102449, 'pass': False}


어휘는 같은데 **IDF 최대 오차가 정확히 1**입니다. "IDF 단계에서 상수 1만큼 어긋났다"는 것이 바로 보입니다. 행렬 오차만 봤다면 원인을 짐작하기 어려웠을 것입니다. 이것이 14.4절에서 **앞 단계부터** 비교하는 이유입니다.

어느 칸이 가장 많이 어긋났는지도 찾아봅니다(4장 `argmax` + `unravel_index`).

In [9]:
bad = NoTailOneTfidf(min_df=3)
x_bad = bad.fit_transform(small)
good = NumpyTfidf(min_df=3)
x_good = good.fit_transform(small)
err = np.abs(x_bad - x_good)
i, j = np.unravel_index(err.argmax(), err.shape)
print(f"가장 큰 차이: 문서 {i}, 단어 '{bad.terms_[j]}' (df={int(bad.df_[j])})")
print(f"  올바른 값 {x_good[i, j]:.4f}  vs  버그 값 {x_bad[i, j]:.4f}")
print(f"  올바른 IDF {good.idf_[j]:.4f}  vs  버그 IDF {bad.idf_[j]:.4f}")

가장 큰 차이: 문서 492, 단어 'time' (df=101)
  올바른 값 0.4790  vs  버그 값 0.3898
  올바른 IDF 2.5916  vs  버그 IDF 1.5916


가장 크게 어긋난 칸은 **흔한 단어**(df가 큰 단어)입니다. +1이 빠지면 흔한 단어의 IDF가 0에 가까워져, 정규화 후 그 단어의 비중이 크게 바뀌기 때문입니다.

### 버그 ② 어휘를 정렬하지 않음

In [10]:
class UnsortedTfidf(NumpyTfidf):
    def fit(self, docs):
        super().fit(docs)
        terms = list(dict.fromkeys(t for doc in docs for t in doc if t in self.vocabulary_))  # 처음 나온 순서
        index = [self.vocabulary_[t] for t in terms]
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.df_, self.idf_ = self.df_[index], self.idf_[index]
        return self


r = compare(small, *base_cfg, min_df=3, vectorizer_cls=UnsortedTfidf)
print("단어→열 사전으로 맞춘 비교:", {k: r[k] for k in ["same_vocab", "max_err", "pass"]})

x_un = UnsortedTfidf(min_df=3).fit_transform(small)
x_sk = TfidfVectorizer(analyzer=identity, lowercase=False, min_df=3).fit_transform(small).toarray()
print("열 순서 그대로 비교         :", f"최대 오차 {np.abs(x_un - x_sk).max():.3f}")

단어→열 사전으로 맞춘 비교: {'same_vocab': True, 'max_err': 9.992007221626409e-16, 'pass': True}


열 순서 그대로 비교         : 최대 오차 0.901


- 우리 `compare` 함수는 **단어 → 열 번호 사전으로 열을 맞춘 뒤** 비교하므로, 어휘 순서가 달라도 값 자체가 맞으면 PASS입니다.
- 열을 맞추지 않고 그대로 빼면 전혀 다른 단어끼리 비교하게 되어 큰 오차가 납니다. 정렬해 두면 이런 실수를 원천적으로 막을 수 있고, 다른 코드(검색 결과 해석, 단어별 가중치)도 단순해집니다.

### 디버깅 순서 정리

| 증상 | 의심할 곳 |
|---|---|
| 어휘 불일치 | 토큰이 다르다: `analyzer`, `lowercase`, `min_df`/`max_df` 해석(정수/실수) |
| 어휘는 같은데 IDF 불일치 | DF 계산(`set(doc)` 빠짐), smooth의 +1, 끝의 +1, 로그 밑(`np.log`는 자연로그) |
| IDF는 같은데 `norm=None` 행렬 불일치 | TF 계산(raw/log), OOV 처리, `np.add.at` 대신 `+=` 사용(중복 누락) |
| `norm=None`은 같은데 `l2`만 불일치 | 정규화 축(`axis=1`), `keepdims`, 0 벡터 처리 |
| 오차가 1e-15 수준 | 정상. 부동소수점 반올림 |

## 📝 README에 쓸 검증 설명

> **sklearn 설정**: `TfidfVectorizer(analyzer=<전처리된 토큰 리스트를 그대로 사용>, lowercase=False, min_df=<영어 3 / 한국어 2>, dtype=float64)`에 `sublinear_tf`, `smooth_idf`, `norm`을 아래 4가지 조합으로 바꿔 가며 비교했습니다. `analyzer`에 함수를 주면 sklearn 자체의 소문자 변환, 토큰 패턴, 불용어가 적용되지 않으므로 두 구현이 정확히 같은 토큰을 봅니다. 어휘 집합이 다르면 행렬을 비교하지 않고 FAIL로 기록하고, 같으면 단어→열 사전으로 열을 맞춘 뒤 최대·평균 절대 오차를 계산해 1e-6 이내면 PASS로 판정했습니다.

| 검증 | 직접 구현 | sklearn | 영어 최대 / 평균 오차 | 한국어 Okt 최대 / 평균 오차 | 결과 |
|---|---|---|---|---|---|
| 1 (기본) | `tf='raw', smooth_idf=True, norm='l2'` | `sublinear_tf=False, smooth_idf=True, norm='l2'` | (14.6 표의 값) | (14.6 표의 값) | PASS |
| 2 | `tf='log'` | `sublinear_tf=True` | | | |
| 3 | `smooth_idf=False` | `smooth_idf=False` | | | |
| 4 | `norm=None` | `norm=None` | | | |

표의 빈칸은 14.6절 결과 표에서 옮겨 적습니다.

## ✍️ 실습: 숨은 버그 찾기

아래 `MysteryTfidf`에는 버그가 하나 있습니다. 코드를 읽지 말고 **검증 결과만으로** 어느 단계가 틀렸는지 좁혀 보세요.
1. 기본 설정(`base_cfg`)으로 `compare`를 실행합니다.
2. `norm=None` 설정으로도 실행해서 결과를 비교합니다.
3. 디버깅 순서 표를 보고 어느 단계가 의심되는지 말한 뒤, 코드를 열어 확인합니다.

In [11]:
class MysteryTfidf(NumpyTfidf):
    def transform(self, docs):
        x = self.count_matrix(docs) * self.idf_
        if self.norm == "l2":
            x /= np.abs(x).sum(axis=1, keepdims=True).clip(min=1e-12)
        return x

In [12]:
# 여기에 작성하세요

**정답**

In [13]:
r1 = compare(small, *base_cfg, min_df=3, vectorizer_cls=MysteryTfidf)
none_cfg = ({"tf": "raw", "smooth_idf": True, "norm": None}, {"sublinear_tf": False, "smooth_idf": True, "norm": None})
r2 = compare(small, *none_cfg, min_df=3, vectorizer_cls=MysteryTfidf)
print("norm='l2':", {k: r1[k] for k in ["same_vocab", "idf_max_err", "max_err", "pass"]})
print("norm=None:", {k: r2[k] for k in ["same_vocab", "idf_max_err", "max_err", "pass"]})

norm='l2': {'same_vocab': True, 'idf_max_err': 0.0, 'max_err': 0.6727178993236484, 'pass': False}
norm=None: {'same_vocab': True, 'idf_max_err': 0.0, 'max_err': 0.0, 'pass': True}


어휘와 IDF는 같고, `norm=None`은 PASS인데 `norm="l2"`만 FAIL입니다. 그러면 **정규화 단계**가 범인입니다. 코드를 보면 제곱합의 제곱근(L2) 대신 절댓값의 합(**L1 노름**)으로 나누고 있습니다. 이렇게 단계를 하나씩 끄고 켜며 비교하면, 코드를 다 읽지 않고도 버그 위치를 빠르게 좁힐 수 있습니다.

## 🧐 퀴즈

**Q1.** `TfidfVectorizer`를 기본 설정으로 쓰면 직접 구현과 비교할 수 없는 이유는?
<details><summary>▶ 정답 보기</summary>

sklearn이 자체 규칙(소문자 변환, 정규식 `\b\w\w+\b`, 불용어 없음)으로 원문을 다시 토큰화해서, 우리 전처리와 다른 어휘가 만들어지기 때문입니다. `analyzer`에 토큰을 그대로 돌려주는 함수를 줘야 같은 토큰을 봅니다.
</details>

**Q2.** 검증에서 행렬보다 어휘와 IDF를 먼저 비교하는 이유는?
<details><summary>▶ 정답 보기</summary>

어휘가 다르면 행렬 비교가 의미가 없고, 앞 단계에서 어긋나면 그 단계가 원인이라는 것을 바로 알 수 있기 때문입니다. 행렬 오차만 보면 어디가 틀렸는지 알기 어렵습니다.
</details>

**Q3.** 최대 오차가 `1.2e-15`로 나왔습니다. 0이 아닌데 PASS로 봐도 되나요?
<details><summary>▶ 정답 보기</summary>

됩니다. 부동소수점 계산은 덧셈 순서 등에 따라 마지막 자리에서 아주 작은 차이가 생깁니다. 허용 오차 1e-6보다 훨씬 작으므로 같은 계산으로 봅니다.
</details>

**Q4.** 기본 설정 하나만 검증하지 않고 4가지 조합을 검증하는 이유는?
<details><summary>▶ 정답 보기</summary>

수식의 각 부분(sublinear TF, smooth IDF, 정규화)이 각각 맞는지 확인하기 위해서입니다. 기본 설정에서 두 버그가 우연히 상쇄되거나, 쓰지 않는 분기(`tf="log"` 등)에 버그가 숨어 있을 수 있습니다.
</details>

**Q5.** README에 sklearn 설정을 적어야 하는 이유는?
<details><summary>▶ 정답 보기</summary>

같은 "TF-IDF"라도 설정에 따라 값이 달라지므로, 어떤 정의로 일치를 확인했는지 적어야 다른 사람이 같은 검증을 재현할 수 있습니다. 과제도 이것을 요구합니다.
</details>

## ✅ 체크포인트
- [ ] 파라미터 대응표를 보지 않고 sklearn 설정을 맞출 수 있다.
- [ ] `analyzer=identity`, `lowercase=False`가 필요한 이유를 설명할 수 있다.
- [ ] 어휘 → IDF → 0 위치 → 값 순서로 비교하는 함수를 만들 수 있다.
- [ ] 4가지 설정 × 데이터별 검증 로그를 만들 수 있다.
- [ ] 검증 실패 시 단계를 바꿔 가며 원인을 좁힐 수 있다.

## 🔗 과제 연결과 다음 장
- `compare`와 `verify_log`는 `src/verify.py`가 되고, 로그는 `outputs/<데이터>/verify_log.txt`로 저장합니다. 과제 요구사항 "Scikit-learn 결과와의 검증 로그 (사용한 sklearn 설정 명시)"에 해당합니다.
- README "검증/실험 결과"에 14.6 표와 위 설정 문장을 씁니다.
- Part 4(TF-IDF 직접 구현)가 끝났습니다. **15장. 코사인 유사도**부터 이 벡터로 문서를 검색합니다.